# Notebook 08 — AI Reviewer Local

Tasks:
- local embedding model
- optional local text-generation model
- discrepancy classification
- evidence summarization
- reviewer explanation

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime

OUTPUT_DIR = "D:/data viewer/data/processed"

## Load Data

In [ ]:
features = pd.read_csv(f"{OUTPUT_DIR}/pairwise_features.csv")
print(f"Total pairs: {len(features)}")
features.head()

## Discrepancy Classification

In [ ]:
def classify_discrepancy(row):
    if row.get('customer_similarity', 1.0) < 0.5:
        return 'ENTITY_MISMATCH'
    if pd.isna(row.get('capacity_diff')) or row.get('capacity_diff', 0) > 2.0:
        return 'NUMERIC_CONFLICT'
    if row.get('status_match', 1.0) == 0:
        return 'STATUS_CONFLICT'
    if row.get('address_similarity', 1.0) < 0.7:
        return 'FORMATTING_DIFFERENCE'
    if row.get('installer_similarity', 1.0) < 0.7:
        return 'POTENTIAL_DUPLICATE'
    return 'OTHER'

features['classification'] = features.apply(classify_discrepancy, axis=1)
print("Classification distribution:")
print(features['classification'].value_counts())

## Evidence Summarization

In [ ]:
def summarize_evidence(row):
    parts = []
    if row.get('customer_similarity', 1.0) < 0.8:
        parts.append(f"Customer name similarity: {row['customer_similarity']:.2f}")
    if row.get('installer_similarity', 1.0) < 0.8:
        parts.append(f"Installer similarity: {row['installer_similarity']:.2f}")
    if row.get('address_similarity', 1.0) < 0.8:
        parts.append(f"Address similarity: {row['address_similarity']:.2f}")
    if row.get('capacity_diff', 0) > 0:
        parts.append(f"Capacity difference: {row['capacity_diff']:.1f} kW")
    if row.get('status_match', 1.0) == 0:
        parts.append('Status mismatch detected')
    return '; '.join(parts) if parts else 'No significant discrepancies'

features['summary'] = features.apply(summarize_evidence, axis=1)
print("Sample summaries:")
print(features['summary'].head(10))

## Clarification Draft

In [ ]:
def draft_clarification(row):
    classification = row.get('classification', 'OTHER')
    if classification == 'NUMERIC_CONFLICT':
        return 'Please verify the system capacity value across sources.'
    elif classification == 'STATUS_CONFLICT':
        return 'Please confirm the current project status.'
    elif classification == 'ENTITY_MISMATCH':
        return 'Please verify if these records refer to the same project.'
    elif classification == 'FORMATTING_DIFFERENCE':
        return 'Please confirm the correct address format.'
    else:
        return 'Please review the flagged discrepancies.'

features['clarification'] = features.apply(draft_clarification, axis=1)
print("Sample clarifications:")
print(features[['pair_id', 'classification', 'clarification']].head(10))

## Explanation

In [ ]:
def explain_decision(row):
    return {
        'what_differs': summarize_evidence(row),
        'why_flagged': f"Classification: {row.get('classification', 'UNKNOWN')}",
        'evidence_considered': [
            f"Customer similarity: {row.get('customer_similarity', 'N/A')}",
            f"Installer similarity: {row.get('installer_similarity', 'N/A')}",
            f"Address similarity: {row.get('address_similarity', 'N/A')}",
            f"Capacity difference: {row.get('capacity_diff', 'N/A')}",
        ],
        'reviewer_action': 'Verify flagged fields and confirm match status.',
    }

features['explanation'] = features.apply(lambda row: json.dumps(explain_decision(row)), axis=1)
features.to_csv(f"{OUTPUT_DIR}/ai_reviewer_output.csv", index=False)
print(f"AI reviewer output saved: {len(features)} rows")